# FLAG 2027 - Notebook 45 RUN-ALL: three face encoders of a different kind than CLIP (research question 1 of 7 Nov)

NB23 showed that bigger models of CLIP's own kind (ViT-H, DFN, PE-Core) do not beat CLIP ViT-L/14 on our task: the face side is saturated for image-text models. NB41 showed that CLIP tuned towards identity while keeping its attribute knowledge moves the needle a little. This notebook tests the three public image encoders that are genuinely different from CLIP and were never run here:

| Block | Model | Why it is different | Output |
|---|---|---|---|
| `siglip2` | SigLIP 2 so400m/14-384 (Google, 2025) | image-text like CLIP but a newer objective, higher resolution and dense self-supervised losses; the strongest open image-text encoder | 1152-d |
| `dino` | DINOv3 ViT-L/16 (Meta, 2025; falls back to DINOv2-L if the gated download is refused) | self-supervised, no text at all: keeps geometry and texture that captions never mention | 1024-d |
| `farl` | FaRL ViT-B/16 (Microsoft, 2022) | CLIP's architecture trained only on 20 M face-caption pairs (LAION-Face): a face-specific CLIP | 512-d |

All three are frozen public models, declared; none sees any test data. Each is extracted on the challenge training rows and the MAV rows and judged in system E's recipe in two forms: added next to CLIP (`F_<block>`) and replacing CLIP (`R_<block>`). Same adoption rule as always.

**How to run:** T4 GPU, Runtime > Run all, about 2.5 hours (downloads about 3 GB, extraction about 40 min, CV about 75 min). DINOv3 is a gated model: accept its terms at https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m with the token account first; if the download is refused the notebook uses DINOv2-L instead and says so. Paste the **SUMMARY**. A winner needs re-extraction on the VoxCeleb parts (faces re-pulled) before it can enter NB20.


In [ ]:
# Cell 1 [NB45 v1 7-Nov] - Drive, GPU (hard check), packages (transformers new enough for SigLIP 2 and DINOv3; OpenAI's clip package for FaRL), HF token (Colab Secrets only)
from google.colab import drive, userdata
drive.mount('/content/drive')
!pip -q install "transformers>=4.56" git+https://github.com/openai/CLIP.git
import torch, shutil, transformers
assert torch.cuda.is_available(), 'NO GPU in this runtime. Runtime > Change runtime type > T4 GPU > Save, then run again.'
dev = 'cuda'
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
print('[NB45 v1 7-Nov] Device:', torch.cuda.get_device_name(0), '| transformers', transformers.__version__, '| HF token found:', bool(HF_TOKEN), '| disk free %.0f GB' % (shutil.disk_usage('/content')[2] / 1e9))


In [ ]:
# Cell 2 [NB45 v1 7-Nov] - face AND voice file lists in our row order (challenge train, MAV v1/v3, MAV extra; dev rows are not needed for CV) - NB39 Cell 2 verbatim
#          + alignment checks: CLIP on 100 faces per split and Whisper layers 20-28 on 48 voices per split must reproduce the stored features
import os, re, json, time, zipfile
import numpy as np, pandas as pd, cv2, librosa
from PIL import Image
from tqdm.auto import tqdm
BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027'); FEATDIR = os.path.join(PROJECT, 'features'); RESDIR = os.path.join(PROJECT, 'results'); CACHE = os.path.join(PROJECT, 'models_cache')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
MAV_ZIPS, MAV_LISTS = os.path.join(BASE, 'dataset'), os.path.join(BASE, 'face_voice_association_splits')
PARTS = os.path.join(FEATDIR, 'nb45_parts'); os.makedirs(PARTS, exist_ok=True)
EXTRACT_NOTES = []
def enote(msg):
    print(msg); EXTRACT_NOTES.append(msg)
# ---- challenge train: faces and voices ----
FULL = '/content/flag_data'; os.makedirs(FULL, exist_ok=True)
if not os.path.exists(os.path.join(FULL, 'train_set.zip.done')):
    with zipfile.ZipFile(os.path.join(ZIP_DIR, 'train_set.zip')) as zf:
        zf.extractall(FULL, members=[m for m in zf.namelist() if m.lower().endswith(('.jpg', '.jpeg', '.png', '.wav', '.txt', '.csv'))])
    open(os.path.join(FULL, 'train_set.zip.done'), 'w').close()
TXT = [os.path.join(r, f) for r, _, fs in os.walk(FULL) for f in fs if f.endswith(('.txt', '.csv'))]
def find_one(*parts):
    hits = [p for p in TXT if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def resolve(rel, list_path):
    q = os.path.dirname(list_path)
    while q.startswith(FULL):
        c = os.path.join(q, rel)
        if os.path.exists(c):
            return c
        q = os.path.dirname(q)
def tok(tokens, exts, fallback=3):
    t = [x for x in tokens if x.lower().endswith(exts)]; return t[0] if len(t) == 1 else tokens[fallback]   # NB35 used token 3 for the face; NB24 the single .wav token
IMG, WAV = ('.jpg', '.jpeg', '.png'), ('.wav',)
SPLITS = ['train', 'mav', 'extra']
lp = find_one('train_English.txt'); lines = [l.split() for l in open(lp) if l.strip()]
FACES = {'train': [resolve(tok(t, IMG), lp) for t in lines]}; WAVS = {'train': [resolve(tok(t, WAV), lp) for t in lines]}
# ---- MAV v1/v3 (same rows as mavceleb_v1v3.npz) ----
def zip_index(zf):
    m = {}
    for n in zf.namelist():
        for key in ('voices/', 'faces/'):
            i = n.find(key)
            if i >= 0 and not n.endswith('/'):
                m[n[i:]] = n; break
    return m
MAVD = '/content/mav'; mav_f, mav_v, mspk = [], [], []
for ver, zname, lists in [('v1', 'mavceleb_v1_train.zip', ['English_train.txt', 'Urdu_train.txt']), ('v3', 'mavceleb_v3_tran.zip', ['English_train.txt', 'German_train.txt'])]:
    flag = os.path.join(MAVD, ver + '.both.done'); need = not os.path.exists(flag)
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); zi = zip_index(zf)
    for lf in lists:
        for l in open(os.path.join(MAV_LISTS, ver, lf)):
            t = l.split()
            if len(t) < 2:
                continue
            vm, fm = zi.get('voices/' + t[0]), zi.get('faces/' + t[1])
            if vm is None or fm is None:
                continue
            if need:
                zf.extract(vm, os.path.join(MAVD, ver)); zf.extract(fm, os.path.join(MAVD, ver))
            mav_v.append(os.path.join(MAVD, ver, vm)); mav_f.append(os.path.join(MAVD, ver, fm)); mspk.append(ver + '_' + t[0].split('/')[0])
    zf.close()
    if need:
        open(flag, 'w').close()
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz')))
assert list(MAV['spk']) == mspk, 'MAV v1/v3 row order differs - tell Claude'
FACES['mav'], WAVS['mav'] = mav_f, mav_v
# ---- MAV v2 + v1-complete 'extra' rows: the SAME random face per clip as the original extraction (same rng sequence as NB24/NB29/NB35) ----
EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz')))
MAX_PER_ID = 120
VPAT = re.compile(r'voices/(id\d+)/([^/]+)/([^/]+)/([^/]+\.wav)$'); FPAT = re.compile(r'faces/(id\d+)/([^/]+)/([^/]+)/([^/]+\.jpg)$')
v1_used = set()
for lf in ['English_train.txt', 'Urdu_train.txt']:
    for l in open(os.path.join(MAV_LISTS, 'v1', lf)):
        t = l.split()
        if t:
            v1_used.add(t[0].split('/')[0])
rows = []
for ver, zname, skip_ids in [('v2', 'mavceleb_v2_complete.zip', set()), ('v1', 'mavceleb_v1_complete.zip', v1_used)]:
    D = '/content/mavx/' + ver; flag = os.path.join(D, 'both.done'); need = not os.path.exists(flag)
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); voices, faces = {}, {}
    for n in zf.namelist():
        mv = VPAT.search(n)
        if mv:
            voices.setdefault(mv.group(1), []).append((mv.group(2), mv.group(3), n)); continue
        mf = FPAT.search(n)
        if mf:
            faces.setdefault((mf.group(1), mf.group(2), mf.group(3)), []).append(n)
    ids_ = sorted(i for i in voices if i not in skip_ids); rng = np.random.default_rng(0)
    for pid in ids_:
        vs = voices[pid]; vs = [vs[j] for j in rng.permutation(len(vs))[:MAX_PER_ID]]
        for lang_, vid, vn in vs:
            fr = faces.get((pid, lang_, vid)) or [f for (p, l, v), fl in faces.items() if p == pid for f in fl]
            if not fr:
                continue
            fn = fr[rng.integers(len(fr))]
            if need:
                zf.extract(vn, D); zf.extract(fn, D)
            rows.append((ver + '_' + pid, os.path.join(D, vn), os.path.join(D, fn)))
    zf.close()
    if need:
        open(flag, 'w').close()
spk_all = np.array([r[0] for r in rows]); keep = np.ones(len(rows), bool)
if len(rows) != len(EXTRA['spk']):
    j = 0; keep[:] = False
    for i in range(len(rows)):
        if j < len(EXTRA['spk']) and spk_all[i] == EXTRA['spk'][j]:
            keep[i] = True; j += 1
    assert j == len(EXTRA['spk']), 'could not align MAV extra rows - tell Claude'
WAVS['extra'] = [r[1] for r, k in zip(rows, keep) if k]; FACES['extra'] = [r[2] for r, k in zip(rows, keep) if k]
assert [r[0] for r, k in zip(rows, keep) if k] == list(EXTRA['spk']), 'MAV extra row order differs - tell Claude'
for s in SPLITS:
    assert None not in FACES[s] and None not in WAVS[s], 'missing face/voice files in %s - tell Claude' % s
    assert len(FACES[s]) == len(WAVS[s])
enote('rows: ' + ', '.join('%s %d' % (s, len(FACES[s])) for s in SPLITS))
def cos_rows(a, b):
    return (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + 1e-9)
# ---- alignment check 1: CLIP on 100 random faces per split must reproduce the stored CLIP rows ----
from transformers import CLIPModel, CLIPProcessor
CLIPM = CLIPModel.from_pretrained('openai/clip-vit-large-patch14').half().to(dev).eval(); CLIPP = CLIPProcessor.from_pretrained('openai/clip-vit-large-patch14')
@torch.no_grad()
def clip_embed(pils):
    x = CLIPP(images=pils, return_tensors='pt')['pixel_values'].to(dev, torch.float16)
    return CLIPM.visual_projection(CLIPM.vision_model(pixel_values=x).pooler_output).float().cpu().numpy()
CLIP_SAVED = np.load(os.path.join(FEATDIR, 'clip.npz'))
stored_f = {'train': CLIP_SAVED['train'], 'mav': MAV['clip'], 'extra': EXTRA['clip']}
rng_c = np.random.default_rng(1)
for s in SPLITS:
    saved = stored_f[s]; assert len(saved) == len(FACES[s]), '%s: %d stored CLIP rows vs %d face files - tell Claude' % (s, len(saved), len(FACES[s]))
    idx = rng_c.choice(len(saved), 100, replace=False)
    med = np.median(cos_rows(clip_embed([Image.open(FACES[s][i]).convert('RGB') for i in idx]), saved[idx]))
    enote('alignment faces  %-6s median CLIP cosine %.4f' % (s, med)); assert med > 0.98, 'face files do not match the stored CLIP rows for %s - tell Claude' % s
# ---- alignment check 2: Whisper layers 20-28 on 48 random voices per split must reproduce the stored whisper rows (same code as NB24) ----
def get_whisper():
    from transformers import WhisperFeatureExtractor, WhisperModel
    name = 'openai/whisper-large-v3'
    return WhisperFeatureExtractor.from_pretrained(name), WhisperModel.from_pretrained(name, torch_dtype=torch.float16).encoder.to(dev).eval()
def whisper_batch(fe, enc, paths, bands):
    """one batch of clips -> one pooled vector per band of layers (mean of layers lo..hi-1, mean over the clip's real frames)"""
    wavs = [librosa.load(p, sr=16000, mono=True)[0][:16000 * 30] for p in paths]
    x = fe(wavs, sampling_rate=16000, return_tensors='pt').input_features.to(dev, torch.float16)
    with torch.no_grad():
        hs = enc(x, output_hidden_states=True).hidden_states
    outs = []
    for lo, hi in bands:
        h = torch.stack(hs[lo:hi]).float().mean(0); o = []
        for k_, w in enumerate(wavs):
            n = max(1, min(1500, len(w) // 320)); o.append(h[k_, :n].mean(0).cpu().numpy())
        outs.append(np.stack(o))
    return outs
WFE, WENC = get_whisper()
WH_SAVED = np.load(os.path.join(FEATDIR, 'whisper.npz'))
rng_w = np.random.default_rng(2)
for s in SPLITS:
    saved = WH_SAVED[s]; assert len(saved) == len(WAVS[s]), '%s: %d stored whisper rows vs %d voice files - tell Claude' % (s, len(saved), len(WAVS[s]))
    idx = rng_w.choice(len(saved), 48, replace=False)
    new = np.concatenate([whisper_batch(WFE, WENC, [WAVS[s][i] for i in idx[b:b + 8]], [(20, 29)])[0] for b in range(0, 48, 8)])
    med = np.median(cos_rows(new, saved[idx])); enote('alignment voices %-6s median Whisper cosine %.4f' % (s, med)); assert med > 0.98, 'voice files do not match the stored whisper rows for %s - tell Claude' % s


In [ ]:
# Cell 3 [NB45 v1c 8-Nov] - three frozen public face encoders of a different kind than CLIP, each embedding every row's canonical face. Resumable per block and split.
#          siglip2: SigLIP 2 so400m/14-384 image features (get_image_features, 1152-d). dino: DINOv3 ViT-L/16 CLS token (gated; DINOv2-L fallback), 1024-d.
#          farl: FaRL-Base-Patch16 (LAION-Face 20M) loaded into OpenAI's CLIP ViT-B/16, encode_image, 512-d. Features -> features/faces2.npz (<block>_<split>).
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoModel, AutoProcessor, AutoImageProcessor
del CLIPM, WFE, WENC; torch.cuda.empty_cache()
POOL = ThreadPoolExecutor(8)
def load_imgs(paths):
    return list(POOL.map(lambda p: Image.open(p).convert('RGB'), paths))
LOADED = {}
def load_siglip2():
    name = 'google/siglip2-so400m-patch14-384'
    m = AutoModel.from_pretrained(name, torch_dtype=torch.float16).to(dev).eval(); pr = AutoProcessor.from_pretrained(name)
    def emb(paths):
        x = pr(images=load_imgs(paths), return_tensors='pt')['pixel_values'].to(dev, torch.float16)
        with torch.no_grad():
            o = m.get_image_features(pixel_values=x)
        if not torch.is_tensor(o):                       # newer transformers wrap the features in a BaseModelOutputWithPooling
            o = o.pooler_output if getattr(o, 'pooler_output', None) is not None else o.last_hidden_state[:, 0]
        return o.float().cpu().numpy()
    return emb, name
def load_dino():
    tried = []
    for name in ['facebook/dinov3-vitl16-pretrain-lvd1689m', 'facebook/dinov2-large']:
        try:
            # v1c: full precision - DINOv3 overflows to inf/NaN in float16 on a T4
            m = AutoModel.from_pretrained(name, torch_dtype=torch.float32, token=HF_TOKEN).to(dev).eval(); pr = AutoImageProcessor.from_pretrained(name, token=HF_TOKEN)
            def emb(paths):
                x = pr(images=load_imgs(paths), return_tensors='pt')['pixel_values'].to(dev, torch.float32)
                with torch.no_grad():
                    o = m(pixel_values=x)
                    e = o.pooler_output if getattr(o, 'pooler_output', None) is not None else o.last_hidden_state[:, 0]
                return e.float().cpu().numpy()
            return emb, name
        except Exception as e:
            tried.append('%s: %s' % (name, repr(e)[:120]))
    raise RuntimeError(' | '.join(tried))
def load_farl():
    import clip, urllib.request
    url = 'https://github.com/FacePerceiver/FaRL/releases/download/pretrained_weights/FaRL-Base-Patch16-LAIONFace20M-ep64.pth'
    dst = os.path.join(CACHE, 'farl', os.path.basename(url)); os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst):
        urllib.request.urlretrieve(url, dst)
    m, pre = clip.load('ViT-B/16', device=dev, jit=False); sd = torch.load(dst, map_location='cpu')
    sd = sd.get('state_dict', sd); res = m.load_state_dict(sd, strict=False)
    miss_vis = [k for k in res.missing_keys if k.startswith('visual.')]
    assert not miss_vis, 'FaRL: %d visual weights missing, e.g. %s' % (len(miss_vis), miss_vis[:3])
    n_vis = sum(k.startswith('visual.') for k in sd); m.eval()
    def emb(paths):
        x = torch.stack([pre(im) for im in load_imgs(paths)]).to(dev)
        with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
            return m.encode_image(x).float().cpu().numpy()
    return emb, 'FaRL-Base-Patch16-LAIONFace20M-ep64 (%d visual tensors loaded, %d unexpected keys ignored)' % (n_vis, len(res.unexpected_keys))
EMBED = {}
for b, loader in [('siglip2', load_siglip2), ('dino', load_dino), ('farl', load_farl)]:
    try:
        EMBED[b], src = loader(); enote('%s: loaded %s' % (b, src))
    except Exception as e:
        enote('%s: NOT loaded (%s) -> skipped' % (b, repr(e)[:200]))
assert EMBED, 'no face encoder could be loaded - tell Claude'
XM = {}; t_all = time.time()
for b in EMBED:
    for s in SPLITS:
        pf = os.path.join(PARTS, '%s_%s.npy' % (b, s))
        if os.path.exists(pf):
            XM[(b, s)] = np.load(pf); continue
        t0 = time.time(); paths = FACES[s]; out = []
        for i in tqdm(range(0, len(paths), 64), desc='%s %s' % (b, s)):
            out.append(EMBED[b](paths[i:i + 64]))
        E = np.concatenate(out).astype(np.float32); assert E.shape[0] == len(paths) and np.isfinite(E).all(), '%s %s: bad output - tell Claude' % (b, s)
        np.save(pf, E); XM[(b, s)] = E; enote('%s %s: %d rows x %d-d in %.0f min' % (b, s, E.shape[0], E.shape[1], (time.time() - t0) / 60))
np.savez(os.path.join(FEATDIR, 'faces2.npz'), **{'%s_%s' % k: v for k, v in XM.items()})
enote('extraction done in %.0f min -> features/faces2.npz (%s)' % ((time.time() - t_all) / 60, ', '.join(EMBED)))
# ---- sanity on challenge rows: identity gap of each block (CLIP 0.24, ArcFace ~0.6; NB35 showed identity-pure face features hurt) ----
Y_CH = np.array([int(t[5]) for t in lines])
def l2n(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def gap(E, lab, n=800, seed=0):
    r = np.random.default_rng(seed); idx = r.choice(len(lab), n, replace=False); Z = l2n(E[idx]); C = Z @ Z.T
    same = (lab[idx][:, None] == lab[idx][None, :]) & ~np.eye(n, dtype=bool); diff = lab[idx][:, None] != lab[idx][None, :]
    return C[same].mean() - C[diff].mean()
for b in EMBED:
    enote('%-8s identity gap (same - different cosine) on challenge rows: %.3f   (CLIP 0.24, LoRA-CLIP 0.48, ArcFace ~0.6 which hurt)' % (b, gap(XM[(b, 'train')], Y_CH)))
EMBED.clear(); torch.cuda.empty_cache(); NOTES45 = list(EXTRACT_NOTES)


In [ ]:
# Cell 4 - features, labels, CV folds, harness (NB36 Cell 2 verbatim, same as NB39 Cell 5)
import os, re, json, time, zipfile, copy
import numpy as np, pandas as pd
import torch.nn as nn, torch.nn.functional as Fn
from sklearn.metrics import roc_curve
BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027'); FEATDIR = os.path.join(PROJECT, 'features'); RESDIR = os.path.join(PROJECT, 'results')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
LISTS = '/content/flag_lists'
if not os.path.exists(LISTS + '/done'):
    with zipfile.ZipFile(os.path.join(ZIP_DIR, 'train_set.zip')) as zf:
        zf.extractall(LISTS, members=[m for m in zf.namelist() if m.endswith(('.txt', '.csv'))])
    open(LISTS + '/done', 'w').close()
TXT = [os.path.join(r, f) for r, _, fs in os.walk(LISTS) for f in fs if f.endswith(('.txt', '.csv'))]
def find_one(*parts):
    hits = [p for p in TXT if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)
Xv, y = read_feat(find_one('train_English_voices.csv'), 192); y = y.astype(int)
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y); id_gender = {c: spk2g[cls2spk[c]] for c in ids}
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz'))); EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz')))
ML = np.load(os.path.join(FEATDIR, 'mav_lang.npz')); EXT_SPK, EXT_LANG = ML['spk'], ML['lang']
assert list(EXT_SPK) == list(MAV['spk']) + list(EXTRA['spk']), 'mav_lang.npz does not match the feature files - tell Claude'
FEAT = {'ecapa': {'train': Xv, 'mav': MAV['ecapa'], 'extra': EXTRA['ecapa']}}
for name in ['clip', 'wavlm_sv', 'r293', 'redim', 'audeer', 'whisper']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); FEAT[name] = {s: d[s] for s in d.files if not s.endswith('_ag')}
FEAT['clip']['mav'], FEAT['clip']['extra'] = MAV['clip'], EXTRA['clip']; FEAT['wavlm_sv']['mav'], FEAT['wavlm_sv']['extra'] = MAV['wavlm_sv'], EXTRA['wavlm_sv']
FEAT['r293']['extra'] = EXTRA['r293']
for name in FEAT:
    for s in ['train', 'mav', 'extra']:
        assert s in FEAT[name] and len(FEAT[name][s]) == {'train': len(y), 'mav': len(MAV['spk']), 'extra': len(EXTRA['spk'])}[s], '%s/%s missing or wrong length - tell Claude' % (name, s)
N_CH = len(y); ext_spk = list(EXT_SPK)
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
Y_ALL = np.r_[y, np.array([ext_lab[s] for s in ext_spk])]; D_ALL = np.r_[np.zeros(N_CH, int), np.ones(len(ext_spk), int)]; EXT = N_CH + np.arange(len(ext_spk))
def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def build(names, split):
    return np.concatenate([l2(FEAT[n][split]) for n in names], 1).astype(np.float32)
def build_train(names):
    return np.concatenate([build(names, 'train'), build(names, 'mav'), build(names, 'extra')])
NAP_K = 8
def nap_basis(V_ext, k):
    diffs = []
    for s in np.unique(EXT_SPK):
        m = EXT_SPK == s; langs = np.unique(EXT_LANG[m])
        if len(langs) < 2:
            continue
        mu = [V_ext[m & (EXT_LANG == l)].mean(0) for l in langs]
        for a in range(len(mu)):
            for b in range(a + 1, len(mu)):
                diffs.append(mu[a] - mu[b])
    Dm = np.stack(diffs); U = np.linalg.svd(Dm - Dm.mean(0), full_matrices=False)[2][:k]
    return U.astype(np.float32), len(diffs)
def nap_apply(X, U):
    return (X - (X @ U.T) @ U).astype(np.float32)
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed); spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5; rng = np.random.default_rng(42); folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))
_PROTO = {}
def max_sep_prototypes(n):
    if n in _PROTO:
        return _PROTO[n]
    P = np.array([[1.0, -1.0]])
    for k in range(2, n):
        Q = np.zeros((k, k + 1)); Q[0, 0] = 1.0; Q[0, 1:] = -1.0 / k; Q[1:, 1:] = np.sqrt(1 - 1.0 / k ** 2) * P; P = Q
    _PROTO[n] = torch.tensor(P.T, dtype=torch.float32); return _PROTO[n]
class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__(); self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class - 1); self.register_buffer('P', max_sep_prototypes(n_class))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv); return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv); a = torch.sigmoid(self.att(torch.cat([f, v], 1))); fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused) @ self.P.t(), ft, vt
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t(); same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
def nce_loss(ft, vt, lab, tau):
    """symmetric identity-aware InfoNCE between the two scored embeddings: every same-identity row counts as a positive"""
    S = Fn.normalize(ft, dim=1) @ Fn.normalize(vt, dim=1).t() / tau; same = lab[:, None] == lab[None, :]
    def one(S_):
        logp = S_ - torch.logsumexp(S_, 1, keepdim=True); return -torch.logsumexp(logp.masked_fill(~same, -1e4), 1).mean()
    return 0.5 * (one(S) + one(S.t()))
@torch.no_grad()
def same_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm; ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()
def dnorm(F_raw, V_raw, tr_idx):
    F, V = F_raw.copy(), V_raw.copy(); ch = None
    for d in np.unique(D_ALL[tr_idx]):
        trd = tr_idx[D_ALL[tr_idx] == d]; alld = np.where(D_ALL == d)[0]
        st = (F_raw[trd].mean(0), F_raw[trd].std(0) + 1e-6, V_raw[trd].mean(0), V_raw[trd].std(0) + 1e-6)
        F[alld] = (F_raw[alld] - st[0]) / st[1]; V[alld] = (V_raw[alld] - st[2]) / st[3]
        if d == 0:
            ch = st
    return F, V, ch
SEEDS = [1, 2, 3]; P_ID, K_CLIP, ALPHA, LR, MAX_EP, PATIENCE, STEPS = 16, 8, 2.0, 1e-4, 100, 15, 60
def train_head(F_raw, V_raw, all_tr, vsets=None, seed=1, cfg=None, max_epochs=MAX_EP):
    """cfg keys: align in {None,'mse','nce'}, w (alignment weight), tau, repair (bool), p_in (input dropout)"""
    cfg = cfg or {}; align, w_al, tau, repair, p_in = cfg.get('align'), cfg.get('w', 1.0), cfg.get('tau', 0.07), cfg.get('repair', False), cfg.get('p_in', 0.0)
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    F, V, ch = dnorm(F_raw, V_raw, all_tr); XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev); model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01); best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), P_ID, replace=False)
            j_np = np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen])
            jv_np = np.concatenate([g[rng.permutation(K_CLIP)] for g in j_np.reshape(P_ID, K_CLIP)]) if repair else j_np   # voice from another clip of the same identity
            j, jv = torch.tensor(j_np, device=dev), torch.tensor(jv_np, device=dev)
            xf = (XF[tr[j]] - mf) / sf_; xv = (XV[tr[jv]] - mv) / sv
            if p_in > 0:
                xf = Fn.dropout(xf, p_in, True); xv = Fn.dropout(xv, p_in, True)
            fused, logits, ft, vt = model(xf, xv); lab = ytr[j]
            loss = Fn.cross_entropy(logits, lab) + ALPHA * opl_loss(fused, lab)
            if align == 'mse':
                loss = loss + w_al * ((ft - vt) ** 2).mean()
            elif align == 'nce':
                loss = loss + w_al * nce_loss(ft, vt, lab, tau)
            opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, XF, XV, ch
R4 = ['ecapa', 'wavlm_sv', 'r293', 'redim', 'audeer']; VOICE = R4 + ['whisper']
F_RAW = build_train(['clip']); V_full = build_train(VOICE); U_NAP, n_d = nap_basis(V_full[N_CH:], NAP_K); V_RAW = nap_apply(V_full, U_NAP)
note('harness ready: %d challenge rows, %d MAV rows, face %d-d, voice %d-d after NAP-%d (%d language-difference vectors)' % (N_CH, len(EXT), F_RAW.shape[1], V_RAW.shape[1], NAP_K, n_d))

In [ ]:
# Cell 5 [NB45 v1 7-Nov] - CV: each new face encoder in system E's recipe (voice R4 + whisper, NAP-8; CE + 2*OPL + 0.5*L_out tau 0.07).
#          F_<b> = block added next to CLIP; R_<b> = block replacing CLIP. R0_ref copied from nb39_cv.csv. Stage A usual folds seeds 1-3; stage B NEW folds + seeds 4-6.
SUMMARY = list(NOTES45) + SUMMARY
d = np.load(os.path.join(FEATDIR, 'faces2.npz'))
FBLOCKS = sorted(set(k.rsplit('_', 1)[0] for k in d.files))
for b in FBLOCKS:
    FEAT[b] = {s: d['%s_%s' % (b, s)] for s in ['train', 'mav', 'extra']}
    assert all(len(FEAT[b][s]) == len(FEAT['clip'][s]) for s in ['train', 'mav', 'extra']), '%s rows do not match - tell Claude' % b
note('face blocks available: ' + ', '.join(FBLOCKS))
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
G_CODE = {gg: i for i, gg in enumerate(sorted(set(id_gender.values())))}; y_g = np.array([G_CODE[id_gender[c]] for c in y])
fold_of = {c: k for k, fl in enumerate(folds) for c in fl}; grp = np.array([fold_of[c] for c in y])
def gender_probe(X):
    acc = []
    for k in range(K):
        tr, te = grp != k, grp == k; clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X[tr], y_g[tr]); acc.append((clf.predict(X[te]) == y_g[te]).mean())
    return 100 * np.mean(acc)
for b in FBLOCKS:
    note('%-8s gender probe on challenge rows (identity-grouped folds): %.1f%%   (CLIP ~99)' % (b, gender_probe(FEAT[b]['train'])))
def lout_loss(ft, vt, lab, tau):
    S = Fn.normalize(ft, dim=1) @ Fn.normalize(vt, dim=1).t() / tau; P = (lab[:, None] == lab[None, :]).float()
    def one(M):
        return -(P * Fn.log_softmax(M, 1)).sum(1).div(P.sum(1)).mean()
    return 0.5 * (one(S) + one(S.t()))
def train_e(F_raw, V_raw, all_tr, vsets, seed, w_al=0.5, tau=0.07):
    """system E's recipe (= NB35 train_lout = NB39 train_e): 16 ids x 8 clips, CE + 2*OPL + 0.5*L_out, early stopping on the fold's pairs"""
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    F, V, ch = dnorm(F_raw, V_raw, all_tr); XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev); model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01); best, best_ep, best_state, wait = 1e9, MAX_EP, None, 0
    for ep in range(1, MAX_EP + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), P_ID, replace=False)
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            fused, logits, ft, vt = model((XF[tr[j]] - mf) / sf_, (XV[tr[j]] - mv) / sv); lab = ytr[j]
            loss = Fn.cross_entropy(logits, lab) + ALPHA * opl_loss(fused, lab) + w_al * lout_loss(ft, vt, lab, tau)
            opt.zero_grad(); loss.backward(); opt.step()
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    model.load_state_dict(best_state); return model, best_ep, XF, XV
COMBOS = {}
for b in FBLOCKS:
    COMBOS['F_' + b] = (['clip', b], VOICE); COMBOS['R_' + b] = ([b], VOICE)
def inputs(fn_, vn_):
    F_raw = build_train(fn_); V_full = build_train(vn_); U, _ = nap_basis(V_full[N_CH:], NAP_K); return F_raw, nap_apply(V_full, U)
rngB = np.random.default_rng(7); foldsB = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rngB.shuffle(members)
    for i, c in enumerate(members):
        foldsB[i % K].append(c)
FOLD_B = []
for k in range(K):
    val_ids = set(foldsB[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_B.append((tr_idx, [make_pairs(va_idx, seed=200 + k), make_pairs(va_idx, same_gender_neg=True, seed=300 + k)]))
CSV45 = os.path.join(RESDIR, 'nb45_cv.csv')
if not os.path.exists(CSV45):
    r = pd.read_csv(os.path.join(RESDIR, 'nb39_cv.csv')); r = r[r.combo == 'R0_ref'][['stage', 'combo', 'seed', 'fold', 'best_ep', 'std', 'gender']].copy()
    assert len(r) == 30, 'R0_ref incomplete in nb39_cv.csv - tell Claude'
    r.to_csv(CSV45, index=False); print('R0_ref copied from NB39 (30 runs)')
done45 = pd.read_csv(CSV45); t0 = time.time()
def run45(stage, names, seeds, fold_data):
    global done45
    for name in names:
        todo = [(seed, k) for seed in seeds for k in range(K) if not ((done45.stage == stage) & (done45.combo == name) & (done45.seed == seed) & (done45.fold == k)).any()]
        if not todo:
            continue
        F_raw, V_raw = inputs(*COMBOS[name])
        for seed in seeds:
            for k in range(K):
                if (seed, k) not in todo:
                    continue
                tr_idx, vsets = fold_data[k]
                try:
                    model, best_ep, XF, XV = train_e(F_raw, V_raw, np.concatenate([tr_idx, EXT]), vsets, seed)
                    e_std, e_gen = [eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets]
                except Exception as e:
                    print(name, 'FAILED', repr(e)[:300]); best_ep, e_std, e_gen = -1, 50.0, 50.0
                row = pd.DataFrame([dict(stage=stage, combo=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen)])
                row.to_csv(CSV45, mode='a', header=False, index=False); done45 = pd.concat([done45, row])
            print('%s %-10s seed %d done | %.0f min' % (stage, name, seed, (time.time() - t0) / 60))
        del F_raw, V_raw
def tab45(stage):
    R = pd.read_csv(CSV45); R = R[R.stage == stage].copy(); R['overall'] = (R['std'] + R['gender']) / 2
    return R.groupby(['combo', 'seed'])[['std', 'gender', 'overall', 'best_ep']].mean().groupby('combo').mean().round(2).sort_values('overall')
def adopt(TB, c):
    refB = TB.loc['R0_ref']; dB, dg, ds = TB.loc[c, 'overall'] - refB['overall'], TB.loc[c, 'gender'] - refB['gender'], TB.loc[c, 'std'] - refB['std']
    return (dB <= -0.4 or (dg <= -0.5 and ds <= 0.2)), dB, ds, dg
run45('A', list(COMBOS), [1, 2, 3], FOLD_DATA)
run45('B', list(COMBOS), [4, 5, 6], FOLD_B)
TA, TB = tab45('A'), tab45('B')
note('NB45 STAGE A (usual folds, seeds 1-3; R0_ref = system E recipe):'); note(TA.to_string())
note('NB45 STAGE B (NEW folds + NEW seeds 4-6):'); note(TB.to_string())
note('VERDICTS (stage B only: overall <= -0.4, or gender <= -0.5 with std <= +0.2 vs R0_ref):')
WIN = []
for c in COMBOS:
    ok, dB, ds, dg = adopt(TB, c)
    note('%-10s %s %s: A %+.2f | B %+.2f (std %+.2f, gender %+.2f) -> %s' % (c, c[2:], 'added next to CLIP' if c[0] == 'F' else 'replacing CLIP', TA.loc[c, 'overall'] - TA.loc['R0_ref', 'overall'], dB, ds, dg, 'ADOPT' if ok else 'reject'))
    if ok:
        WIN.append(c)
if WIN:
    best_c = min(WIN, key=lambda c: TB.loc[c, 'overall'])
    note('NEXT: ADOPTED (%s); best %s. Claude builds NB45b: the winning encoder on the VoxCeleb parts test + 1 (faces re-pulled from the mp4 zips), then the NB20 system.' % (', '.join(WIN), best_c))
else:
    note('NEXT: nothing adopted; SigLIP 2, DINO and FaRL add nothing to CLIP on this task.')
print('=' * 70); print('SUMMARY  (copy everything between the lines and paste to Claude)'); print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)
